# Games Backtest (2026 default)

Backtest variant of `games_today_tomorrow.ipynb`:

- Defaults to **completed 2026 games** from `data/games_2026.parquet`.
- Keeps V6-style scoring logic, but changes data selection upstream.
- If Kalshi historical implied columns are missing, scoring still runs and Kalshi-based metrics are omitted (set to NaN).

Run top-to-bottom after refreshing Parquet files.

In [1]:
# Backtest parameters (edit these)
SEASON = 2026
START_DATE = "2026-03-25"  # inclusive, YYYY-MM-DD
END_DATE = None             # inclusive; None = today

FINAL_STATES = {"Final", "Game Over", "Completed Early"}

print({
    "SEASON": SEASON,
    "START_DATE": START_DATE,
    "END_DATE": END_DATE or "today",
    "FINAL_STATES": sorted(FINAL_STATES),
})

{'SEASON': 2026, 'START_DATE': '2026-03-25', 'END_DATE': 'today', 'FINAL_STATES': ['Completed Early', 'Final', 'Game Over']}


In [2]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"
PARQUET = DATA / f"games_{SEASON}.parquet"

if not PARQUET.is_file():
    raise FileNotFoundError(f"Missing {PARQUET}. Run: python -m app.main live --season {SEASON}")

df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
df["detailed_state"] = df["detailed_state"].astype(str)

df_bt = df[df["detailed_state"].isin(FINAL_STATES)].copy()

if "home_win" in df_bt.columns:
    # ensure final training label is present for backtest
    df_bt = df_bt[df_bt["home_win"].notna()].copy()

start_ts = pd.Timestamp(START_DATE).normalize()
end_ts = pd.Timestamp.today().normalize() if END_DATE is None else pd.Timestamp(END_DATE).normalize()
if end_ts < start_ts:
    raise ValueError(f"END_DATE ({end_ts.date()}) must be >= START_DATE ({start_ts.date()})")

sub = df_bt[df_bt["game_date"].between(start_ts, end_ts)].copy()

print("Using:", PARQUET.resolve())
print(f"Rows total={len(df)}  completed={len(df_bt)}")
if len(df_bt):
    print(f"Date range completed: {df_bt['game_date'].min().date()} .. {df_bt['game_date'].max().date()}")
print(f"Backtest window: {start_ts.date()} .. {end_ts.date()}  rows={len(sub)}")


Using: C:\Users\Austin\baseball\mlb-model\data\games_2026.parquet
Rows total=2796  completed=597
Date range completed: 2026-03-01 .. 2026-04-14
Backtest window: 2026-03-25 .. 2026-04-15  rows=279


In [3]:
# Core thresholds from your mismatch workflow
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2

USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

favorites = s.copy().sort_values(["game_date", "home_team_name"]).reset_index(drop=True)
favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = np.where(
    favorites["_mismatch_side"].eq("home_favored"),
    favorites["home_team_name"],
    favorites["away_team_name"],
)

print("Favorites frame rows:", len(favorites))
display(favorites.tail(25))

Favorites frame rows: 279


,game_pk,game_date,detailed_state,home_team_name,away_team_name,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher,away_probable_pitcher,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,sp_kbb_diff,sp_xfip_diff,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_fip,away_pen_season_fip,home_pen_season_era,away_pen_season_era,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,_mismatch_side,core_matches,favored_team
254,825024,2026-04-13,Final,Athletics,Texas Rangers,133,140,1.0,8.0,Luis Severino,Nathan Eovaldi,622663,543135,R,R,92.902673,97.754733,9.302326,19.587629,4.444828,4.207692,0.0,True,2026,2026,-10.285303,0.237135,-4.852060,97.272983,104.723081,-7.450098,10.000000,14.285714,3.340000,4.800000,4.588550,3.607042,4.534351,1.901408,4.271429,3.740777,4.371429,2.359223,-0.317121,0.133734,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Texas Rangers
255,824937,2026-04-13,Final,Atlanta Braves,Miami Marlins,144,146,4.0,10.0,Grant Holmes,Eury Pérez,656550,691587,R,R,111.454667,104.319285,7.954545,9.890110,4.115385,5.650000,0.0,True,2026,2026,-1.935564,-1.534615,7.135382,111.329772,111.189204,0.140568,8.163265,4.545455,3.968421,7.100000,2.238614,3.156075,0.801980,3.532710,2.449398,3.740449,0.975904,4.247191,0.210784,0.584375,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Miami Marlins
256,824856,2026-04-13,Final,Baltimore Orioles,Arizona Diamondbacks,110,109,9.0,7.0,Dean Kremer,Ryne Nelson,665152,669194,R,R,104.747407,96.470364,42.857143,12.790698,7.300000,4.968852,1.0,True,2026,2026,30.066445,2.331148,8.277043,102.333427,90.525724,11.807703,NaN,11.111111,NaN,4.938710,3.563415,4.040299,3.731707,4.634328,3.874194,3.274757,3.774194,4.194175,0.310779,-0.765541,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Arizona Diamondbacks
257,823964,2026-04-13,Final,Los Angeles Dodgers,New York Mets,119,121,4.0,0.0,Justin Wrobleski,David Peterson,680736,656849,L,L,119.589002,89.192275,1.587302,11.340206,3.276471,3.303390,1.0,True,2026,2026,-9.752905,-0.026919,30.396727,129.824910,97.144847,32.680064,-2.631579,14.893617,3.877778,2.028571,3.289189,3.247887,2.675676,1.711268,3.850000,2.977551,3.907895,1.377551,0.560811,-0.270336,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,New York Mets
258,823725,2026-04-13,Final,Minnesota Twins,Boston Red Sox,142,111,13.0,6.0,Bailey Ober,Garrett Crochet,641927,676979,R,L,104.747407,96.898487,10.588235,16.304348,3.913559,4.573684,1.0,True,2026,2026,-5.716113,-0.660125,7.848920,108.784322,92.071971,16.712351,7.142857,24.000000,2.789655,3.100000,3.824138,4.240496,3.956897,3.123967,4.045652,3.841176,4.402174,3.494118,0.221514,-0.399319,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Boston Red Sox
259,823563,2026-04-13,Final,New York Yankees,Los Angeles Angels,147,108,11.0,10.0,Will Warren,Yusei Kikuchi,701542,579328,R,L,98.040148,103.320331,17.500000,10.112360,3.481818,4.822222,1.0,True,2026,2026,7.387640,-1.340404,-5.280183,72.373657,98.678662,-26.305005,18.181818,16.666667,4.358065,2.035484,2.000000,3.803448,2.700000,3.351724,1.996552,3.871429,3.724138,3.342857,-0.003448,0.067980,NaN,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,NaN,away_favored,NaN,Los Angeles Angels
260,823481,2026-04-13,Final,Philadelphia Phillies,Chicago Cubs,143,112,13.0,7.0,Cristopher Sánchez,Javier Assad,650911,665871,L,R,99.467225,100.894301,25.263158,4.444444,1.846269,5.700000,1.0,True,2026,2026,20.818713,-3.853731,-1.4270

In [4]:
# V6 model block (kept aligned in spirit with games_today_tomorrow)
s = favorites.copy()

# Stable scaling
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
PLATOON_ABS = 10.0
PEN_ABS = 0.75

kbb_norm = s["sp_kbb_diff"] / SP_KBB_ABS
xfip_norm = -s["sp_xfip_diff"] / SP_XFIP_ABS
off_norm = s["offense_diff"] / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

# Pen: prefer roll14; neutralize missing values so matrix math is stable
if "home_pen_roll14_fip" in s.columns:
    hr_pen = s["home_pen_roll14_fip"]
    if "home_pen_season_fip" in s.columns:
        hr_pen = hr_pen.combine_first(s["home_pen_season_fip"])
else:
    hr_pen = pd.Series(np.nan, index=s.index)

if "away_pen_roll14_fip" in s.columns:
    ar_pen = s["away_pen_roll14_fip"]
    if "away_pen_season_fip" in s.columns:
        ar_pen = ar_pen.combine_first(s["away_pen_season_fip"])
else:
    ar_pen = pd.Series(np.nan, index=s.index)

pen_gap = hr_pen - ar_pen  # home - away; lower home FIP better
pen_norm = (-pen_gap / PEN_ABS).fillna(0.0)

sp_vector = (0.65 * kbb_norm) + (0.35 * xfip_norm)
off_vector = (0.70 * off_norm) + (0.30 * platoon_norm)
pen_vector = pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])
sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

agreement = 1 - np.mean(np.var(sign_matrix, axis=0))
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (0.45 * agreement) + (0.30 * direction_purity) + (0.25 * mag_consistency)
raw_edge = mean_mag
instability = np.std(signal_matrix, axis=0)

direction_conflict = (
    (np.sign(sp_vector) != np.sign(off_vector)).astype(int)
    + (np.sign(sp_vector) != np.sign(pen_vector)).astype(int)
    + (np.sign(off_vector) != np.sign(pen_vector)).astype(int)
)

risk_penalty = (
    0.35 * direction_conflict
    + 0.45 * instability
    + 0.20 * np.abs(sp_vector - off_vector)
)
risk_penalty = np.tanh(risk_penalty)

trap_score = raw_edge * (1 - coherence_score)
quality_score = raw_edge * coherence_score * np.exp(-1.25 * instability) * (1 / (1 + risk_penalty))
risk_adjusted_edge = quality_score - 0.5 * trap_score

prefer_home = sp_vector >= 0
s["favored_team"] = np.where(prefer_home, s["home_team_name"], s["away_team_name"])
s["_mismatch_side"] = np.where(prefer_home, "home_favored", "away_favored")

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = direction_conflict
s["instability"] = instability
s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score
s["risk_penalty"] = risk_penalty

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int)
    + (np.abs(xfip_norm) > 1).astype(int)
    + (np.abs(off_norm) > 1).astype(int)
)

# Backtest outcome: did the model's favored side win?
if "home_win" in s.columns:
    s["favorite_won"] = np.where(
        prefer_home,
        s["home_win"] == 1.0,
        s["home_win"] == 0.0,
    )
else:
    s["favorite_won"] = np.nan

# Kalshi is optional in historical backtests.
if {"kalshi_home_implied", "kalshi_away_implied"}.issubset(s.columns):
    s["implied_prob"] = np.where(prefer_home, s["kalshi_home_implied"], s["kalshi_away_implied"])
    s["market_edge"] = s["risk_adjusted_edge"] - s["implied_prob"]
else:
    s["implied_prob"] = np.nan
    s["market_edge"] = np.nan

scored = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False]).reset_index(drop=True)

show_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "_mismatch_side", "favorite_won",
    "risk_adjusted_edge", "quality_score", "coherence_score", "instability",
    "signal_agreement", "signal_conflict", "core_matches",
    "sp_kbb_diff", "sp_xfip_diff", "offense_diff", "offense_platoon_diff",
    "implied_prob", "market_edge", "home_win",
]
show_cols = [c for c in show_cols if c in scored.columns]

print(f"Scored games: {len(scored)}")
display(scored[show_cols].head(25))

# Optional aggregate view (uncomment to use):
# display(
#     scored["favorite_won"]
#     .map({True: "Favorite Won", False: "Favorite Did Not Win"})
#     .fillna("Unknown")
#     .value_counts(dropna=False)
#     .rename_axis("favorite_result")
#     .to_frame("count")
# )

Scored games: 279


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,favorite_won,risk_adjusted_edge,quality_score,coherence_score,instability,signal_agreement,signal_conflict,core_matches,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,implied_prob,market_edge,home_win
0,2026-04-01,Los Angeles Dodgers,Cleveland Guardians,Los Angeles Dodgers,home_favored,False,0.659980,0.926877,0.704713,0.153564,3,0,3,4.340627,-0.928762,19.550947,18.414394,NaN,NaN,0.0
1,2026-04-01,Atlanta Braves,Athletics,Atlanta Braves,home_favored,True,0.529442,0.851982,0.686695,0.258167,3,0,2,8.675203,0.246082,18.551993,28.361209,NaN,NaN,1.0
2,2026-04-07,Minnesota Twins,Detroit Tigers,Minnesota Twins,home_favored,True,0.367355,0.494612,0.711787,0.117098,3,0,1,2.580645,-0.552582,6.421844,8.981117,NaN,NaN,1.0
3,2026-04-07,Los Angeles Angels,Atlanta Braves,Atlanta Braves,away_favored,True,0.235856,0.433724,0.671116,0.365202,0,3,0,-2.982879,0.367384,-8.134335,-13.985485,NaN,NaN,0.0
4,2026-04-08,Minnesota Twins,Detroit Tigers,Minnesota Twins,home_favored,True,0.234269,0.325306,0.706471,0.144279,3,0,2,4.819005,0.896893,6.421844,8.981117,NaN,NaN,1.0
5,2026-04-06,Minnesota Twins,Detroit Tigers,Minnesota Twins,home_favored,True,0.227623,0.379119,0.682730,0.283785,3,0,1,1.922121,-0.828125,6.421844,4.638741,NaN,NaN,1.0
6,2026-04-09,Miami Marlins,Cincinnati Reds,Miami Marlins,home_favored,True,0.131945,0.345202,0.659655,0.456352,3,0,2,5.253623,0.109997,15.269717,27.129604,NaN,NaN,1.0
7,2026-04-12,Baltimore Orioles,San Francisco Giants,Baltimore Orioles,home_favored,True,0.125416,0.197968,0.690142,0.236712,3,0,1,1.576577,0.148649,13.271810,-4.510982,NaN,NaN,1.0
8,2026-04-13,Atlanta Braves,Miami Marlins,Atlanta Braves,home_favored,False,0.082179,0.250116,0.650127,0.541929,3,0,1,-1.935564,-1.534615,7.135382,0.140568,NaN,NaN,0.0
9,2026-04-03,Los Angeles Angels,Seattle Mariners,Los Angeles Angels,home_favored,False,0.074954,0.240973,0.650690,0.536593,3,0,2,5.812678,0.218235,11.273904,24.215868,NaN,NaN,0.0


In [5]:
# Additional breakdowns by metric bands
band_df = scored.copy()

# Keep known outcomes for cleaner win-rate summaries
band_df = band_df[band_df["favorite_won"].isin([True, False])].copy()

band_specs = {
    "risk_adjusted_edge": [-np.inf, 0.25, 0.50, 0.75, 1.00, np.inf],
    "quality_score": [-np.inf, 0.10, 0.20, 0.30, 0.40, np.inf],
    "coherence_score": [-np.inf, 0.40, 0.50, 0.60, 0.70, np.inf],
    "instability": [-np.inf, 0.80, 1.00, 1.20, 1.50, np.inf],
}

for metric, bins in band_specs.items():
    if metric not in band_df.columns:
        continue

    d = band_df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        continue

    d["band"] = pd.cut(d[metric], bins=bins, include_lowest=True)

    out = (
        d.groupby("band", observed=False)["favorite_won"]
        .agg(
            games="count",
            favorite_wins=lambda x: int((x == True).sum()),
            favorite_losses=lambda x: int((x == False).sum()),
            favorite_win_rate="mean",
        )
        .reset_index()
    )
    out["favorite_win_rate"] = out["favorite_win_rate"].round(3)

    print(f"\n{metric} band breakdown")
    display(out)



risk_adjusted_edge band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.25]",276,152,124,0.551
1,"(0.25, 0.5]",1,1,0,1.000
2,"(0.5, 0.75]",2,1,1,0.500
3,"(0.75, 1.0]",0,0,0,NaN
4,"(1.0, inf]",0,0,0,NaN



quality_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.1]",218,123,95,0.564
1,"(0.1, 0.2]",39,18,21,0.462
2,"(0.2, 0.3]",8,4,4,0.500
3,"(0.3, 0.4]",9,6,3,0.667
4,"(0.4, inf]",5,3,2,0.600



coherence_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.4]",149,84,65,0.564
1,"(0.4, 0.5]",36,20,16,0.556
2,"(0.5, 0.6]",55,27,28,0.491
3,"(0.6, 0.7]",36,21,15,0.583
4,"(0.7, inf]",3,2,1,0.667



instability band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.8]",35,18,17,0.514
1,"(0.8, 1.0]",21,13,8,0.619
2,"(1.0, 1.2]",19,11,8,0.579
3,"(1.2, 1.5]",31,20,11,0.645
4,"(1.5, inf]",173,92,81,0.532


In [6]:
# Decision / confidence layers + backtest summary
def decision_layer(df: pd.DataFrame) -> pd.Series:
    conditions = [
        (df["risk_adjusted_edge"] > 1.0) & (df["coherence_score"] > 0.60) & (df["instability"] < 1.2),
        (df["risk_adjusted_edge"] > 0.55) & (df["coherence_score"] > 0.45),
    ]
    choices = ["BET", "LEAN"]
    return np.select(conditions, choices, default="PASS")


def confidence_tier(df: pd.DataFrame) -> pd.Series:
    return np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.75),
        "A (Strong Bet)",
        np.where(
            df["decision"] == "BET",
            "B (Playable Bet)",
            np.where(df["decision"] == "LEAN", "C (Lean Only)", "D (Pass)"),
        ),
    )


bt = scored.copy()
bt["decision"] = decision_layer(bt)
bt["tier"] = confidence_tier(bt)

# Evaluate pick correctness only on non-PASS rows
bt_eval = bt[bt["decision"].isin(["BET", "LEAN"])].copy()
bt_eval["pick_home"] = bt_eval["_mismatch_side"].eq("home_favored")
bt_eval["won"] = np.where(bt_eval["pick_home"], bt_eval["home_win"] == 1.0, bt_eval["home_win"] == 0.0)

summary = {
    "rows_scored": int(len(bt)),
    "rows_bet_or_lean": int(len(bt_eval)),
    "win_rate_bet_or_lean": float(bt_eval["won"].mean()) if len(bt_eval) else np.nan,
    "bets_only": int((bt["decision"] == "BET").sum()),
    "leans_only": int((bt["decision"] == "LEAN").sum()),
}

print(summary)
if len(bt_eval):
    by_tier = bt_eval.groupby("tier")["won"].agg(["count", "mean"]).rename(columns={"mean": "win_rate"})
    display(by_tier.sort_values("count", ascending=False))

view_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "decision", "tier",
    "risk_adjusted_edge", "coherence_score", "instability", "home_win", "won",
]
view_cols = [c for c in view_cols if c in bt_eval.columns]
display(bt_eval.sort_values("risk_adjusted_edge", ascending=False)[view_cols].head(50))

{'rows_scored': 279, 'rows_bet_or_lean': 1, 'win_rate_bet_or_lean': 0.0, 'bets_only': 0, 'leans_only': 1}


,count,win_rate
tier,,
C (Lean Only),1,0.0


,game_date,home_team_name,away_team_name,favored_team,decision,tier,risk_adjusted_edge,coherence_score,instability,home_win,won
0,2026-04-01,Los Angeles Dodgers,Cleveland Guardians,Los Angeles Dodgers,LEAN,C (Lean Only),0.65998,0.704713,0.153564,0.0,False


In [7]:
# Quantile calibration tables + monotonicity checks

cal_df = scored.copy()
if "favorite_won" not in cal_df.columns:
    raise ValueError("favorite_won missing; run V6 scoring cell first.")

cal_df = cal_df[cal_df["favorite_won"].isin([True, False])].copy()
if cal_df.empty:
    raise ValueError("No rows with known favorite_won values for calibration.")

metrics = [
    "risk_adjusted_edge",
    "quality_score",
    "coherence_score",
    "instability",
]


def quantile_calibration_table(df: pd.DataFrame, metric: str, q: int = 10, ascending_good: bool = True):
    d = df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        return None, None

    # Handle low-variance metrics safely.
    n_unique = d[metric].nunique(dropna=True)
    q_eff = max(2, min(q, int(n_unique)))

    d["quantile"] = pd.qcut(d[metric], q=q_eff, duplicates="drop")
    tab = (
        d.groupby("quantile", observed=False)["favorite_won"]
        .agg(games="count", favorite_wins="sum", favorite_win_rate="mean")
        .reset_index()
    )
    tab["favorite_losses"] = tab["games"] - tab["favorite_wins"]
    tab["favorite_win_rate"] = tab["favorite_win_rate"].round(3)

    rates = tab["favorite_win_rate"].to_numpy()
    if not ascending_good:
        rates = rates[::-1]
    deltas = np.diff(rates)
    mono_score = float((deltas >= 0).mean()) if len(deltas) else np.nan
    is_monotonic = bool((deltas >= 0).all()) if len(deltas) else True

    summary = {
        "metric": metric,
        "quantile_bins": int(len(tab)),
        "rows_used": int(tab["games"].sum()),
        "win_rate_first_bin": float(tab.iloc[0]["favorite_win_rate"]),
        "win_rate_last_bin": float(tab.iloc[-1]["favorite_win_rate"]),
        "lift_last_minus_first": float(tab.iloc[-1]["favorite_win_rate"] - tab.iloc[0]["favorite_win_rate"]),
        "is_monotonic_expected_direction": is_monotonic,
        "monotonicity_fraction": round(mono_score, 3) if not np.isnan(mono_score) else np.nan,
    }
    return tab, summary


# For instability, lower values are generally better; reverse expected direction.
expected_ascending = {
    "risk_adjusted_edge": True,
    "quality_score": True,
    "coherence_score": True,
    "instability": False,
}

summaries = []
for m in metrics:
    if m not in cal_df.columns:
        continue
    tab, s = quantile_calibration_table(cal_df, m, q=10, ascending_good=expected_ascending[m])
    if tab is None:
        continue
    print(f"\n{m}: quantile calibration")
    display(tab)
    summaries.append(s)

if summaries:
    print("\nMonotonicity summary")
    display(pd.DataFrame(summaries).sort_values("metric").reset_index(drop=True))


risk_adjusted_edge: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-2.335, -1.03]",28,18,0.643,10
1,"(-1.03, -0.811]",28,17,0.607,11
2,"(-0.811, -0.675]",28,15,0.536,13
3,"(-0.675, -0.551]",35,16,0.457,19
4,"(-0.551, -0.479]",21,11,0.524,10
5,"(-0.479, -0.377]",27,17,0.630,10
6,"(-0.377, -0.308]",28,15,0.536,13
7,"(-0.308, -0.198]",28,13,0.464,15
8,"(-0.198, -0.049]",28,17,0.607,11
9,"(-0.049, 0.66]",28,15,0.536,13



quality_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-0.0006399999999999999, 0.00705]",28,17,0.607,11
1,"(0.00705, 0.0158]",28,16,0.571,12
2,"(0.0158, 0.0216]",29,12,0.414,17
3,"(0.0216, 0.0314]",27,15,0.556,12
4,"(0.0314, 0.0414]",28,19,0.679,9
5,"(0.0414, 0.0557]",27,20,0.741,7
6,"(0.0557, 0.0742]",28,15,0.536,13
7,"(0.0742, 0.11]",28,12,0.429,16
8,"(0.11, 0.178]",28,10,0.357,18
9,"(0.178, 0.927]",28,18,0.643,10



coherence_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.229, 0.342]",28,17,0.607,11
1,"(0.342, 0.356]",28,17,0.607,11
2,"(0.356, 0.365]",28,13,0.464,15
3,"(0.365, 0.376]",28,11,0.393,17
4,"(0.376, 0.39]",28,21,0.750,7
5,"(0.39, 0.418]",27,14,0.519,13
6,"(0.418, 0.561]",28,19,0.679,9
7,"(0.561, 0.585]",30,11,0.367,19
8,"(0.585, 0.618]",26,16,0.615,10
9,"(0.618, 0.712]",28,15,0.536,13



instability: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.1, 0.727]",28,16,0.571,12
1,"(0.727, 1.008]",28,15,0.536,13
2,"(1.008, 1.356]",28,17,0.607,11
3,"(1.356, 1.54]",28,18,0.643,10
4,"(1.54, 1.677]",28,8,0.286,20
5,"(1.677, 2.038]",27,18,0.667,9
6,"(2.038, 2.325]",28,10,0.357,18
7,"(2.325, 2.728]",28,16,0.571,12
8,"(2.728, 3.446]",28,18,0.643,10
9,"(3.446, 6.175]",28,18,0.643,10



Monotonicity summary


,metric,quantile_bins,rows_used,win_rate_first_bin,win_rate_last_bin,lift_last_minus_first,is_monotonic_expected_direction,monotonicity_fraction
0,coherence_score,10,279,0.607,0.536,-0.071,False,0.444
1,instability,10,279,0.571,0.643,0.072,False,0.444
2,quality_score,10,279,0.607,0.643,0.036,False,0.444
3,risk_adjusted_edge,10,279,0.643,0.536,-0.107,False,0.333
